In [1]:
!pip -q install peft sentencepiece

In [2]:
import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

### Load Dataset

In [3]:
alpaca = load_dataset("yahma/alpaca-cleaned")
print(alpaca)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [4]:
small_data = alpaca["train"].shuffle(seed=42).select(range(2500))
split = small_data.train_test_split(test_size=0.1, seed=42)

train_raw = split["train"]
val_raw = split["test"]

print("Train:", len(train_raw))
print("Val  :", len(val_raw))

Train: 2250
Val  : 250


In [7]:
sample = train_raw[0]

print("Instruction:\n", sample["instruction"])
print("\nInput:\n", sample["input"])
print("\nOutput:\n", sample["output"])

Instruction:
 What is the value of x if 
    x    = y+5,
    y    = z+10,
    z    = w+20,
and  w = 80?


Input:
 

Output:
 Plugging the known value of w into the third given equation, we find that z=100. Plugging z into the second given equation, we find that y=110. Plugging y into the first given equation gives x=115.


### Load Model

In [8]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "right"

print(tokenizer.pad_token)
print(tokenizer.eos_token)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<|endoftext|>
<|im_end|>
